Replication of the 2017 paper: Attention Is All You Need per the tutorial here: 

https://nlp.seas.harvard.edu/2018/04/03/attention.html


### Library versions from original
pytorch 0.4.1 <br>
numpy=1.15.4 matplotlib=2.2.3 seaborn=0.8.1 cython=0.29 <br>
spacy==2.1.8 <br>

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, copy, time
from torch.autograd import Variable
import matplotlib.pyplot as plt
import seaborn
seaborn.set_context(context="talk")
%matplotlib inline

# Model Build Starts Here

In [3]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many 
    other models.
    """
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)
    
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)
    
        
    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask,
                            tgt, tgt_mask)
    

In [4]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)

In [5]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [6]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [7]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
        
    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [8]:
class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        return x + self.dropout(sublayer(self.norm(x)))

In [9]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

In [10]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."
    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
        
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

In [11]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
 
    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

In [12]:
def subsequent_mask(size):
    "Mask out subsequent positions."
    attn_shape = (1, size, size)
    subsequent_mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
    return torch.from_numpy(subsequent_mask) == 0

In [13]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) \
             / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = F.softmax(scores, dim = -1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

In [14]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        "Take in model size and number of heads."
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        # We assume d_v always equals d_k
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
        
    def forward(self, query, key, value, mask=None):
        "Implements Figure 2"
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        
        # 1) Do all the linear projections in batch from d_model => h x d_k 
        query, key, value = \
            [l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
             for l, x in zip(self.linears, (query, key, value))]
        
        # 2) Apply attention on all the projected vectors in batch. 
        x, self.attn = attention(query, key, value, mask=mask, 
                                 dropout=self.dropout)
        
        # 3) "Concat" using a view and apply a final linear. 
        x = x.transpose(1, 2).contiguous() \
             .view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)

In [15]:
class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))

In [16]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

In [17]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) *
                             -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + Variable(self.pe[:, :x.size(1)], 
                         requires_grad=False)
        return self.dropout(x)
        

In [18]:
def make_model(src_vocab, tgt_vocab, N=6, 
               d_model=512, d_ff=2048, h=8, dropout=0.1):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), 
                             c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab))
    
    # This was important from their code. 
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform(p)
    return model

In [19]:
def make_model(src_vocab, tgt_vocab, N=6, 
               d_model=512, d_ff=2048, h=8, dropout=0.1):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), 
                             c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab))
    
    # This was important from their code. 
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform(p)
    return model

In [20]:
tmp_model = make_model(10, 10, 2)

c:\Users\JC\miniconda3\envs\torch03\lib\site-packages\ipykernel_launcher.py:20: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.


# Training Setup Starts Here

In [21]:
class Batch:
    "Object for holding a batch of data with mask during training."
    def __init__(self, src, trg=None, pad=0):
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)
        if trg is not None:
            self.trg = trg[:, :-1]
            self.trg_y = trg[:, 1:]
            self.trg_mask = \
                self.make_std_mask(self.trg, pad)
            self.ntokens = (self.trg_y != pad).data.sum()
    
    @staticmethod
    def make_std_mask(tgt, pad):
        "Create a mask to hide padding and future words."
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & Variable(
            subsequent_mask(tgt.size(-1)).type_as(tgt_mask.data))
        return tgt_mask

In [22]:
def run_epoch(data_iter, model, loss_compute):
    "Standard Training and Logging Function"
    start = time.time()
    total_tokens = 0
    total_loss = 0
    tokens = 0
    for i, batch in enumerate(data_iter):
        
        out = model.forward(batch.src, batch.trg, 
                            batch.src_mask, batch.trg_mask)
        loss = loss_compute(out, batch.trg_y, batch.ntokens)
        total_loss += loss
        total_tokens += batch.ntokens
        tokens += batch.ntokens
        if i % 50 == 1:

            elapsed = time.time() - start
            print("Epoch Step: %d Loss: %f Tokens per Sec: %f" % (i, (loss.float() / batch.ntokens.float()).item(),tokens.float().item() / elapsed))
            start = time.time()
            tokens = 0
    return total_loss.float() / total_tokens.float()

In [23]:
global max_src_in_batch, max_tgt_in_batch
def batch_size_fn(new, count, sofar):
    "Keep augmenting batch and calculate total number of tokens + padding."
    global max_src_in_batch, max_tgt_in_batch
    if count == 1:
        max_src_in_batch = 0
        max_tgt_in_batch = 0
    max_src_in_batch = max(max_src_in_batch,  len(new.src))
    max_tgt_in_batch = max(max_tgt_in_batch,  len(new.trg) + 2)
    src_elements = count * max_src_in_batch
    tgt_elements = count * max_tgt_in_batch
    return max(src_elements, tgt_elements)

In [24]:
class NoamOpt:
    "Optim wrapper that implements rate."
    def __init__(self, model_size, factor, warmup, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size
        self._rate = 0
        
    def step(self):
        "Update parameters and rate"
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p['lr'] = rate
        self._rate = rate
        self.optimizer.step()
        
    def rate(self, step = None):
        "Implement `lrate` above"
        if step is None:
            step = self._step
        return self.factor * \
            (self.model_size ** (-0.5) *
            min(step ** (-0.5), step * self.warmup ** (-1.5)))
        
def get_std_opt(model):
    return NoamOpt(model.src_embed[0].d_model, 2, 4000,
            torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))

In [25]:
class LabelSmoothing(nn.Module):
    "Implement label smoothing."
    def __init__(self, size, padding_idx, smoothing=0.0):
        super(LabelSmoothing, self).__init__()
        self.criterion = nn.KLDivLoss(size_average=False)
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None
        
    def forward(self, x, target):
        assert x.size(1) == self.size
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.numel() > 0:
            
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        self.true_dist = true_dist
        return self.criterion(x, Variable(true_dist, requires_grad=False))

In [26]:
def data_gen(V, batch, nbatches):
    "Generate random data for a src-tgt copy task."
    for i in range(nbatches):
        data = torch.randint(1, V, (batch, 10))  # random token indices
        src = data.long()  # cast here
        tgt = data.long()  # cast here
        yield Batch(src, tgt, 0)

In [27]:
class SimpleLossCompute:
    "A simple loss compute and train function."
    def __init__(self, generator, criterion, opt=None):
        self.generator = generator
        self.criterion = criterion
        self.opt = opt
        
    def __call__(self, x, y, norm):
        norm = norm.float()
        x = self.generator(x)
        loss = self.criterion(x.contiguous().view(-1, x.size(-1)), 
                              y.contiguous().view(-1)) / norm
        loss.backward()
        if self.opt is not None:
            self.opt.step()
            self.opt.optimizer.zero_grad()
        return loss.data[0] * norm

In [28]:
V = 11
criterion = LabelSmoothing(size=V, padding_idx=0, smoothing=0.0)
model = make_model(V, V, N=2)


model_opt = NoamOpt(model.src_embed[0].d_model, 1, 400,
        torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))

for epoch in range(10):
    model.train()
    run_epoch(data_gen(V, 30, 20), model, 
              SimpleLossCompute(model.generator, criterion, model_opt))
    model.eval()
    print(run_epoch(data_gen(V, 30, 5), model, 
                    SimpleLossCompute(model.generator, criterion, None)))

c:\Users\JC\miniconda3\envs\torch03\lib\site-packages\torch\nn\functional.py:52: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))
c:\Users\JC\miniconda3\envs\torch03\lib\site-packages\ipykernel_launcher.py:20: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.
c:\Users\JC\miniconda3\envs\torch03\lib\site-packages\ipykernel_launcher.py:17: UserWarning: invalid index of a 0-dim tensor. This will be an error in PyTorch 0.5. Use tensor.item() to convert a 0-dim tensor to a Python number


Epoch Step: 1 Loss: 3.068969 Tokens per Sec: 257.065557
Epoch Step: 1 Loss: 1.985791 Tokens per Sec: 1316.680111
tensor(1.9912)
Epoch Step: 1 Loss: 1.958302 Tokens per Sec: 952.321033
Epoch Step: 1 Loss: 1.723083 Tokens per Sec: 1377.543621
tensor(1.7432)
Epoch Step: 1 Loss: 1.838574 Tokens per Sec: 1063.204413
Epoch Step: 1 Loss: 1.665912 Tokens per Sec: 1386.331210
tensor(1.6772)
Epoch Step: 1 Loss: 1.764614 Tokens per Sec: 1031.480171
Epoch Step: 1 Loss: 1.554952 Tokens per Sec: 1392.829217
tensor(1.5350)
Epoch Step: 1 Loss: 1.938676 Tokens per Sec: 1049.545858
Epoch Step: 1 Loss: 1.235950 Tokens per Sec: 1401.514280
tensor(1.2573)
Epoch Step: 1 Loss: 1.457588 Tokens per Sec: 1051.008403
Epoch Step: 1 Loss: 0.960453 Tokens per Sec: 1354.220412
tensor(0.9271)
Epoch Step: 1 Loss: 1.171500 Tokens per Sec: 1060.516632
Epoch Step: 1 Loss: 0.742116 Tokens per Sec: 1349.973929
tensor(0.7070)
Epoch Step: 1 Loss: 0.825220 Tokens per Sec: 1069.702122
Epoch Step: 1 Loss: 0.407575 Tokens per Se

In [ ]:
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    memory = model.encode(src, src_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data)
    for i in range(max_len-1):
        out = model.decode(memory, src_mask, 
                           Variable(ys), 
                           Variable(subsequent_mask(ys.size(1))
                                    .type_as(src.data)))
        prob = model.generator(out[:, -1])
        print(prob)
        _, next_word = torch.max(prob, dim = 1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, 
                        torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=1)
    return ys

model.eval()
src = Variable(torch.LongTensor([[1,2,3,5,5,6,7,8,10,10]]) )
src_mask = Variable(torch.ones(1, 1, 10) )
print(greedy_decode(model, src, src_mask, max_len=10, start_symbol=1))

tensor([[-12.2113,  -5.4633,  -0.0751,  -3.9554,  -4.7573,  -9.0470,  -4.5971,
          -6.8152,  -3.6757,  -5.7837,  -7.3222]],
       grad_fn=<LogSoftmaxBackward>)
tensor([[-10.8770,  -2.4021,  -3.0261,  -0.3826,  -5.2418,  -6.2078,  -3.8359,
          -5.9002,  -2.0103,  -4.4989,  -6.1509]],
       grad_fn=<LogSoftmaxBackward>)
tensor([[-12.4914,  -5.8052,  -6.3202,  -6.3669,  -7.3189,  -0.0335,  -4.0260,
          -5.9664,  -6.2561,  -5.9586,  -7.0731]],
       grad_fn=<LogSoftmaxBackward>)
tensor([[-13.5206,  -6.2863,  -5.4119,  -7.7697,  -7.9584,  -2.7458,  -0.0811,
          -6.1899,  -6.6937,  -6.3177,  -6.5105]],
       grad_fn=<LogSoftmaxBackward>)
tensor([[-10.8994,  -4.1788,  -4.1412,  -5.2471,  -6.5934,  -0.4535,  -1.7153,
          -2.0956,  -5.6157,  -4.7462,  -4.4679]],
       grad_fn=<LogSoftmaxBackward>)
tensor([[-12.3758,  -5.4113,  -4.3143,  -6.6267,  -6.7409,  -4.0260,  -2.3355,
          -0.1834,  -5.1189,  -5.5742,  -3.7807]],
       grad_fn=<LogSoftmaxBackward>

In [38]:
model = torch.load('iwslt.pt')

c:\Users\JC\miniconda3\envs\torch03\lib\site-packages\torch\serialization.py:425: SourceChangeWarning: source code of class 'torch.nn.modules.container.ModuleList' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
c:\Users\JC\miniconda3\envs\torch03\lib\site-packages\torch\serialization.py:425: SourceChangeWarning: source code of class 'torch.nn.modules.linear.Linear' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  warnings.warn(msg, SourceChangeWarning)
c:\Users\JC\miniconda3\envs\torch03\lib\site-packages\torch\cuda\__init__.py:114: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 3050 Ti Laptop GPU which requires CUDA_VERSION >= 9000 for
     optimal performance and

In [37]:
def beam_search_decode(model, src, src_mask, max_len, start_symbol,
                       beam_size=5, alpha=0.6):

    memory = model.encode(src, src_mask)

    # beam elements: (sequence_tensor, log_prob)
    beams = [(torch.ones(1,1).fill_(start_symbol).type_as(src.data), 0.0)]

    for i in range(max_len - 1):

        new_beams = []

        for seq, score in beams:

            out = model.decode(
                memory,
                src_mask,
                Variable(seq),
                Variable(subsequent_mask(seq.size(1)).type_as(src.data))
            )

            prob = model.generator(out[:, -1])

            topk_log_probs, topk_ids = torch.topk(prob, beam_size)

            #build new beams individually for each of the top k candidates
            for k in range(beam_size):

                next_word = topk_ids[0][k].item()
                next_score = score + topk_log_probs[0][k].item()

                new_seq = torch.cat([
                    seq,
                    torch.ones(1,1).type_as(src.data).fill_(next_word)
                ], dim=1)

                new_beams.append((new_seq, next_score))

        # length penalty (Transformer paper)
        def length_penalty(length):
            return ((5 + length) ** alpha) / ((5 + 1) ** alpha)

        # rank beams
        new_beams = sorted(
            new_beams,
            key=lambda x: x[1] / length_penalty(x[0].size(1)),
            reverse=True
        )
        
        # print(f"Top {beam_size} beams:")
        # for seq, score in new_beams[:beam_size]:
        #     print(f"  {seq}: {score}")

        beams = new_beams[:beam_size]

    return beams[0][0]



model.eval()
src = Variable(torch.LongTensor([[1,2,3,5,5,6,7,8,10,10]]) )
src_mask = Variable(torch.ones(1, 1, 10) )
print(beam_search_decode(model, src, src_mask, max_len=10, start_symbol=1))

tensor([[ 1,  2,  3,  5,  6,  5,  7,  8, 10, 10]])


In [72]:
import json
import re
from collections import defaultdict

class BPETokenizer:

    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = []
        self.vocab = []
        self.word_freqs = None


    def pre_tokenize(self, text):
        return re.findall(r"\w+|[^\w\s]", text)


    def build_word_freqs(self, corpus):

        word_freqs = defaultdict(int)

        for sentence in corpus:
            words = self.pre_tokenize(sentence)
            for w in words:
                word_freqs[w] += 1

        self.word_freqs = word_freqs


    def compute_pair_freqs(self, splits):

        pair_freqs = defaultdict(int)

        for word, freq in self.word_freqs.items():

            split = splits[word]

            for i in range(len(split)-1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += freq

        return pair_freqs


    def merge_pair(self, pair, splits):

        a, b = pair
        new_symbol = a + b

        for word in splits:

            split = splits[word]
            i = 0

            while i < len(split)-1:

                if split[i] == a and split[i+1] == b:
                    split = split[:i] + [new_symbol] + split[i+2:]
                else:
                    i += 1

            splits[word] = split

        return splits


    def train(self, corpus):

        print("Building word frequencies...")
        self.build_word_freqs(corpus)

        splits = {}

        for word in self.word_freqs:
            splits[word] = list(word) + ["</w>"]

        alphabet = set()

        for word in self.word_freqs:
            for c in word:
                alphabet.add(c)

        alphabet = sorted(alphabet)

        self.vocab = ["<blank>", "<s>", "</s>"] + alphabet + ["</w>"]

        print("Training BPE...")

        while len(self.vocab) < self.vocab_size:

            pair_freqs = self.compute_pair_freqs(splits)

            if not pair_freqs:
                break

            best = max(pair_freqs, key=pair_freqs.get)

            self.merges.append(best)
            self.vocab.append(best[0] + best[1])

            #builds updated splits for all words wuth the new merged symbol
            splits = self.merge_pair(best, splits)

        print("BPE training finished")
        print("Vocab size:", len(self.vocab))


    def tokenize(self, text):

        words = self.pre_tokenize(text)

        splits = [list(word) + ["</w>"] for word in words]
        
        for pair in self.merges:

            a, b = pair
            merge = a + b

            for idx, split in enumerate(splits):

                i = 0

                while i < len(split)-1:

                    if split[i] == a and split[i+1] == b:
                        split = split[:i] + [merge] + split[i+2:]
                    else:
                        i += 1

                splits[idx] = split

        tokens = []

        for split in splits:
            
            split = [s.replace("</w>","") for s in split]

            for i, token in enumerate(split):

                if i < len(split)-1:
                    tokens.append(token + "@@")
                else:
                    tokens.append(token)

        return tokens
    
    def detokenize(self, tokens):

        sentence = " ".join(tokens)

        sentence = sentence.replace("@@ ", "")

        return sentence


    def save(self, path):

        with open(path, "w") as f:
            json.dump({
                "vocab": self.vocab,
                "merges": self.merges
            }, f)


    def load(self, path):

        with open(path) as f:
            tok = json.load(f)

        self.vocab = tok["vocab"]
        self.merges = [tuple(m) for m in tok["merges"]]

In [4]:
from torchtext import datasets


# train, val, test = datasets.IWSLT.splits(exts=('.de','.en'), fields=(None,None), root=".data/")
train, val, test = datasets.WMT14.splits(exts=('.de','.en'), fields=(None,None))

# corpus = []

# for ex in train.examples:
#     corpus.append(ex.src)
#     corpus.append(ex.trg)

# tokenizer = BPETokenizer(vocab_size=25)

# tokenizer.train(corpus)

# tokenizer.save("bpe_tokenizer.json")

FileNotFoundError: [Errno 2] No such file or directory: '.data\\newstest2013.tok.bpe.32000.de'

In [73]:
tokenizer = BPETokenizer()

tokenizer.load("bpe_tokenizer.json")

tokenizer.detokenize(tokenizer.tokenize("I am snug like a bug in a rug"))

'I am snug like a bug in a rug'